# 07 — Prédiction sur Hanoï

Objectif :
1. Télécharger carte + bâtiments OSMnx des zones d'étude
2. Extraire les **mêmes features de morphologie que le notebook 04** (méthodo du papier)
3. Charger le modèle pré-entraîné (notebook 06), le calibrer avec nos mesures terrain
4. Prédire le niveau de bruit sur toute la zone
5. Exporter la carte de bruit (CSV + GeoJSON)

In [ ]:
import osmnx as ox
import geopandas as gpd
import pandas as pd
import numpy as np
import joblib
import folium
from shapely.geometry import Point
import warnings
warnings.filterwarnings('ignore')

# Zones d'étude Hanoï
STUDY_AREA = 'Bach Khoa, Hanoi, Vietnam'  # ajuste selon besoin
GRAPH_PATH = '../data/processed/hanoi_roads.graphml'

In [ ]:
import os

if not os.path.exists(GRAPH_PATH):
    print(f'Téléchargement carte : {STUDY_AREA}...')
    G = ox.graph_from_place(STUDY_AREA, network_type='drive')
    ox.save_graphml(G, GRAPH_PATH)
    print('Sauvegardé.')
else:
    G = ox.load_graphml(GRAPH_PATH)

nodes, edges = ox.graph_to_gdfs(G)
print(f'{len(edges)} segments routiers')

# Affiche la zone
ax = edges.plot(figsize=(10, 10), linewidth=0.5, color='gray')
ax.set_title(STUDY_AREA)
import matplotlib.pyplot as plt
plt.tight_layout()
plt.savefig('../outputs/maps/hanoi_osm.png', dpi=150)
plt.show()

In [ ]:
# Génère une grille de points sur la zone d'étude
bounds = edges.total_bounds  # [minx, miny, maxx, maxy]
GRID_STEP = 0.0003  # ~30 mètres

lons = np.arange(bounds[0], bounds[2], GRID_STEP)
lats = np.arange(bounds[1], bounds[3], GRID_STEP)
grid_points = [(lat, lon) for lat in lats for lon in lons]
print(f'Grille : {len(grid_points)} points')

grid_df = pd.DataFrame(grid_points, columns=['latitude', 'longitude'])

In [ ]:
# Features de morphologie pour chaque point de la grille
# — mêmes métriques que le notebook 04 (papier : R=300 m, normalisé πR²)
R = 300
AREA_KM2 = np.pi * (R / 1000) ** 2
CRS_HANOI = 'EPSG:32648'  # UTM 48N

# Bâtiments de la zone
buildings = ox.features_from_place(STUDY_AREA, tags={'building': True})
buildings = buildings[buildings.geometry.geom_type.isin(['Polygon', 'MultiPolygon'])]
print(f'{len(buildings)} bâtiments')

pts = gpd.GeoDataFrame(
    grid_df.copy(),
    geometry=gpd.points_from_xy(grid_df.longitude, grid_df.latitude),
    crs='EPSG:4326'
).to_crs(CRS_HANOI)

bld = buildings.to_crs(CRS_HANOI)
bld['geometry'] = bld.geometry.centroid
edg = edges.to_crs(CRS_HANOI)
nod = nodes.to_crs(CRS_HANOI)

buf = pts[['geometry']].copy()
buf['geometry'] = buf.geometry.buffer(R)
buf['pt_id'] = range(len(buf))

# Building density
jb = gpd.sjoin(bld[['geometry']], buf, predicate='within').groupby('pt_id').size()
pts['building_density_km2'] = pts.index.map(lambda i: jb.get(i, 0) / AREA_KM2)

# Road density (km/km²)
jr = gpd.sjoin(edg[['geometry']], buf, predicate='intersects')
rl = jr.groupby('pt_id').apply(lambda g: g.geometry.length.sum())
pts['road_density_km_km2'] = pts.index.map(lambda i: (rl.get(i, 0) / 1000) / AREA_KM2)

# Intersection count
jn = gpd.sjoin(nod[['geometry']], buf, predicate='within').groupby('pt_id').size()
pts['intersection_count'] = pts.index.map(lambda i: jn.get(i, 0))

# Distance à la route
pts['dist_road_m'] = pts.geometry.apply(lambda p: edg.distance(p).min())

grid_df = pd.DataFrame(pts.drop(columns='geometry'))
grid_df['hour'] = 8       # heure cible de la prédiction
grid_df['is_weekend'] = 0
grid_df.to_parquet('../data/processed/hanoi_grid_features.parquet', index=False)
print('Features grille sauvegardées.')

In [ ]:
# Charge le modèle pré-entraîné sur Sunbird (notebook 06)
model = joblib.load('../outputs/models/surrogate_lgbm.pkl')

FEATURES = ['building_density_km2', 'road_density_km_km2', 'intersection_count',
            'dist_road_m', 'hour', 'is_weekend']
grid_df['noise_pred_dB'] = model.predict(grid_df[FEATURES])

print(grid_df['noise_pred_dB'].describe())

In [ ]:
# ============================================================
#  TRANSFER LEARNING : modèle Uganda → données Hanoï
#  Tourne dès que data/raw/hanoi/measurements.csv existe (notebook 09)
# ============================================================
import os
import lightgbm as lgb

MEASURES = '../data/raw/hanoi/measurements.csv'

if os.path.exists(MEASURES):
    hanoi = pd.read_csv(MEASURES, parse_dates=['timestamp'])

    # --- 1. Mêmes features de morphologie pour NOS points mesurés ---
    hpts = gpd.GeoDataFrame(
        hanoi, geometry=gpd.points_from_xy(hanoi.longitude, hanoi.latitude),
        crs='EPSG:4326').to_crs(CRS_HANOI)
    hbuf = hpts[['geometry']].copy()
    hbuf['geometry'] = hbuf.geometry.buffer(R)
    hbuf['pt_id'] = range(len(hbuf))

    jb = gpd.sjoin(bld[['geometry']], hbuf, predicate='within').groupby('pt_id').size()
    hpts['building_density_km2'] = hpts.index.map(lambda i: jb.get(i, 0) / AREA_KM2)
    jr = gpd.sjoin(edg[['geometry']], hbuf, predicate='intersects')
    rl = jr.groupby('pt_id').apply(lambda g: g.geometry.length.sum())
    hpts['road_density_km_km2'] = hpts.index.map(lambda i: (rl.get(i, 0) / 1000) / AREA_KM2)
    jn = gpd.sjoin(nod[['geometry']], hbuf, predicate='within').groupby('pt_id').size()
    hpts['intersection_count'] = hpts.index.map(lambda i: jn.get(i, 0))
    hpts['dist_road_m'] = hpts.geometry.apply(lambda p: edg.distance(p).min())

    X_hanoi = pd.DataFrame(hpts.drop(columns='geometry'))[FEATURES]
    y_hanoi = hanoi['noise_dB']

    # --- 2. Split train/test sur NOS points pour mesurer honnêtement ---
    from sklearn.model_selection import train_test_split
    Xh_tr, Xh_te, yh_tr, yh_te = train_test_split(
        X_hanoi, y_hanoi, test_size=0.3, random_state=42)

    # --- 3. Baseline : modèle Uganda brut (avant adaptation) ---
    mae_raw = (yh_te - model.predict(Xh_te)).abs().mean()

    # --- 4a. CALIBRATION par offset (simple, robuste avec peu de points) ---
    offset = (yh_tr - model.predict(Xh_tr)).mean()
    mae_offset = (yh_te - (model.predict(Xh_te) + offset)).abs().mean()

    # --- 4b. FINE-TUNING : on CONTINUE l'entraînement du modèle Uganda
    #         sur nos points Hanoï (init_model = transfer learning LightGBM) ---
    finetuned = lgb.train(
        {'objective': 'regression', 'learning_rate': 0.01,
         'num_leaves': 15, 'verbose': -1},
        lgb.Dataset(Xh_tr, yh_tr),
        num_boost_round=100,
        init_model=model.booster_,          # <-- on part du modèle pré-entraîné
    )
    mae_ft = (yh_te - finetuned.predict(Xh_te)).abs().mean()

    print(f'MAE Uganda brut        : {mae_raw:5.1f} dB')
    print(f'MAE + offset (4a)      : {mae_offset:5.1f} dB   (offset = {offset:+.1f} dB)')
    print(f'MAE fine-tuné (4b)     : {mae_ft:5.1f} dB')

    # --- 5. On garde la meilleure méthode et on applique à la grille ---
    if mae_ft <= mae_offset:
        print('>> Fine-tuning retenu')
        grid_df['noise_pred_dB'] = finetuned.predict(grid_df[FEATURES])
        finetuned.save_model('../outputs/models/surrogate_lgbm_hanoi.txt')
    else:
        print('>> Calibration par offset retenue (plus robuste avec peu de points)')
        grid_df['noise_pred_dB'] = model.predict(grid_df[FEATURES]) + offset
else:
    print('Pas encore de mesures Hanoï — la carte reste sur le modèle Uganda brut.')
    print('Lance le notebook 09 après ta première session terrain.')

In [ ]:
# Export pour GAMA et visualisation
output = grid_df[['latitude', 'longitude', 'noise_pred_dB']]
output.to_csv('../outputs/maps/hanoi_noise_map.csv', index=False)

gdf = gpd.GeoDataFrame(
    output,
    geometry=gpd.points_from_xy(output.longitude, output.latitude),
    crs='EPSG:4326'
)
gdf.to_file('../outputs/maps/hanoi_noise_map.geojson', driver='GeoJSON')
print('Exports sauvegardés.')

# Carte interactive
m = folium.Map(location=[grid_df.latitude.mean(), grid_df.longitude.mean()], zoom_start=15)
folium.plugins.HeatMap(
    data=grid_df[['latitude', 'longitude', 'noise_pred_dB']].values.tolist(),
    radius=15, blur=10, min_opacity=0.4
).add_to(m)
m.save('../outputs/maps/hanoi_heatmap.html')
m